## Study takeaway (read first)

**Topic:** ReAct agent demo with Gemini API (Lectures 9 & 12)

**Walkthrough:** (1) Plain LLM fails multi-step task. (2) Define tools. (3) Manual ReAct loop: Thought->Action->Observation. (4) SDK auto-orchestration.

**Remember for exam:** ReAct loop; agents use tools for real actions; needs GEMINI_API_KEY in Colab secrets.

**Time:** ~30min | **Exam priority:** Low

### 1. Setup & Authentication (Read First!)

Welcome to the Agentic AI Lab! Before we can build our agentic systems, you need to authenticate your notebook with the Gemini API. We will do this  securely using Colab's built-in Secret Manager so you never expose your password in the code.

**Step 1: Get your FREE API Key**
1. Navigate to [Google AI Studio](https://aistudio.google.com/).
2. Sign in with your standard Google account.
3. Click on **"Get API key"** in the left-hand navigation menu.
4. Click **"Create API key"** and copy the generated string to your clipboard. (You may need to choose a project for that key; you can click "create new project" if none are existing already.)

**Step 2: Add the Key to Colab Secrets**
1. Look at the left sidebar of this Colab window and click the 🔑 **Secrets** icon (it looks like a key).
2. Click **"Add new secret"**.
3. In the "Name" field, type exactly: `GEMINI_API_KEY`
4. In the "Value" field, paste the API key you just copied.
5. **Crucial:** Toggle the switch next to your new secret to turn on **Notebook access**.

Once you have completed these steps, run the code cell below to install the necessary libraries and securely load your environment!

In [ ]:
# Install the Google GenAI SDK and Yahoo Finance
!pip install -q -U google-genai yfinance

import os
import yfinance as yf
from google import genai
from google.genai import types
from google.colab import userdata

# Load the API key
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
    print("✅ Environment successfully authenticated.")
except Exception as e:
    print("❌ Error: Could not find GEMINI_API_KEY in Colab Secrets.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.7/793.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.5/246.5 kB 19.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.50.0 which is incompatible.
✅ Environment successfully authenticated.


### 2. Phase 1: The Limitation (Standard LLM)
*Demonstrating how a standard, reactive LLM fails when asked to perform real-time data retrieval. The numbers are likely hallucinated!*

In [ ]:
prompt = (
    "I want to buy 5 shares of NVIDIA right now. "
    "Exactly how much will that cost me in Swiss Francs (CHF) "
    "based on the current live market?"
)

print(f"User: {prompt}\n")
print("--- Standard LLM Response ---")

# A single, reactive generation pass without tools
response_standard = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=prompt
)
print(response_standard.text)

User: I want to buy 5 shares of NVIDIA right now. Exactly how much will that cost me in Swiss Francs (CHF) based on the current live market?

--- Standard LLM Response ---
To give you the most accurate estimate, I am using the live market data as of **Friday, May 24, 2024, at approximately 10:30 AM ET.**

Because NVIDIA (NVDA) is currently extremely volatile following its recent earnings report, these numbers change every few seconds.

### 1. The Share Price (USD)
*   **Current NVDA Price:** ~$1,058.50 USD per share.
*   **Cost for 5 Shares:** $5,292.50 USD.

### 2. The Conversion (CHF)
*   **Current USD/CHF Exchange Rate:** ~0.9145
*   **Conversion:** $5,292.50 × 0.9145 = **4,840.00 CHF**

---

### Total Estimated Cost: **4,840.00 CHF**

---

### Important Factors to Consider:
1.  **The "Spread":** When you buy a stock, you pay the "Ask" price, which is usually a few cents higher than the "Last" price shown on Google or news sites.
2.  **Brokerage Fees:** Depending on whether you use 

### 3. Phase 2: Equipping the Agent (Defining Tools)
*Providing the LLM with "hands." These Python functions allow the agent to fetch live market data and calculate currency exchange rates.*

In [ ]:
def get_live_stock_price(ticker_symbol: str) -> float:
    """Gets the current live price of a stock in USD."""
    print(f"⚙️ AGENT ACTION: Fetching live stock price for {ticker_symbol}...")
    try:
        ticker = yf.Ticker(ticker_symbol)
        price = ticker.fast_info['lastPrice']
        print(f"👀 AGENT OBSERVATION: {ticker_symbol} is currently ${price:.2f}")
        return round(price, 2)
    except Exception as e:
        print("⚠️ Network error. Using fallback data.")
        return 125.00

def convert_currency(amount: float, from_currency: str, to_currency: str) -> float:
    """Converts an amount from one currency to another using Yahoo Finance."""
    print(f"⚙️ AGENT ACTION: Converting {amount} from {from_currency} to {to_currency}...")
    try:
        # Construct the Yahoo Finance currency pair (e.g., "USDCHF=X")
        pair = f"{from_currency}{to_currency}=X"
        ticker = yf.Ticker(pair)

        # Fetch the latest exchange rate
        exchange_rate = ticker.fast_info['lastPrice']
        converted_amount = amount * exchange_rate

        print(f"👀 AGENT OBSERVATION: Conversion successful -> {converted_amount:.2f} {to_currency}")
        return round(converted_amount, 2)
    except Exception as e:
        print("⚠️ Network error or invalid currency pair. Using fallback data.")
        return round(amount * 0.90, 2)

# Group the tools
agent_tools = [get_live_stock_price, convert_currency]
print("✅ Tools successfully loaded into memory.")

✅ Tools successfully loaded into memory.


### Testing the Tools Manually
*Before we let the AI orchestrate these tools, let's verify they work as expected. Notice that they return native Python floats, which is the best practice when passing data back to an LLM.*

In [ ]:
print("--- Testing the Stock API ---")
# Let's test a random stock, e.g., Apple
aapl_price = get_live_stock_price("AAPL")
print(f"Data passed to LLM: {aapl_price} (Data type: {type(aapl_price).__name__})\n")

print("--- Testing the Currency API ---")
# Let's convert $100 to Euros
chf_amount = convert_currency(100.0, "USD", "CHF")
print(f"Data passed to LLM: {chf_amount} (Data type: {type(chf_amount).__name__})\n")

print("✅ Tools are functioning correctly. We are ready for the Agent.")

--- Testing the Stock API ---
⚙️ AGENT ACTION: Fetching live stock price for AAPL...
👀 AGENT OBSERVATION: AAPL is currently $287.51
Data passed to LLM: 287.51 (Data type: float)

--- Testing the Currency API ---
⚙️ AGENT ACTION: Converting 100.0 from USD to CHF...
👀 AGENT OBSERVATION: Conversion successful -> 77.76 CHF
Data passed to LLM: 77.76 (Data type: float)

✅ Tools are functioning correctly. We are ready for the Agent.


### 4. Phase 3: The Agentic Loop (Manual Orchestration)
*Under the hood of an Agent: Using the ReAct (Reasoning + Acting) framework to manually orchestrate thoughts, actions, and observations until the goal is met.*

In [ ]:
def run_agent(user_prompt: str):
    """Executes a manual ReAct loop for a given prompt."""

    # 1. Initialize the stateless conversation history
    messages = [
        types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)])
    ]

    print("==================================================")
    print(f"User: {user_prompt}")
    print("==================================================\n")
    print("--- Agentic System Execution Trace ---")

    # 2. The Manual Orchestration Loop
    while True:
        # Send the history to the model.
        # We explicitly disable automatic_function_calling to retain manual control.
        response = client.models.generate_content(
            model="gemini-3-flash-preview",
            contents=messages,
            config=types.GenerateContentConfig(
                tools=agent_tools,
                automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
            )
        )

        # Append the model's raw response to the history
        messages.append(response.candidates[0].content)

        # 3. Check if the model requested a tool
        if response.function_calls:
            # Grab the requested tool
            tool_call = response.function_calls[0]
            tool_name = tool_call.name
            tool_args = tool_call.args

            print(f"\n⚙️ EXECUTING TOOL: '{tool_name}'...")

            # 4. The Routing Logic
            if tool_name == "get_live_stock_price":
                result = get_live_stock_price(**tool_args)
            elif tool_name == "convert_currency":
                result = convert_currency(**tool_args)
            else:
                result = "Error: Tool not recognized."

            # 5. Package observation and append to history for the next loop
            function_response_part = types.Part.from_function_response(
                name=tool_name,
                response={"result": result}
            )
            messages.append(
                types.Content(role="user", parts=[function_response_part])
            )

        else:
            # 6. Goal achieved! Break the loop and print the final answer.
            print("\n--- Final Output to User ---")
            final_answer = "".join(
                p.text for p in response.candidates[0].content.parts
                if p.text and not getattr(p, 'thought', False)
            )
            print(final_answer.strip() + "\n")
            break

In [ ]:
# Test 1: The original NVIDIA scenario
run_agent(
    "I want to buy 5 shares of NVIDIA right now. "
    "Exactly how much will that cost me in Swiss Francs (CHF)?"
)

User: I want to buy 5 shares of NVIDIA right now. Exactly how much will that cost me in Swiss Francs (CHF)?

--- Agentic System Execution Trace ---

⚙️ EXECUTING TOOL: 'get_live_stock_price'...
⚙️ AGENT ACTION: Fetching live stock price for NVDA...
👀 AGENT OBSERVATION: NVDA is currently $207.83

⚙️ EXECUTING TOOL: 'convert_currency'...
⚙️ AGENT ACTION: Converting 1039.15 from USD to CHF...
👀 AGENT OBSERVATION: Conversion successful -> 808.23 CHF

--- Final Output to User ---
5 shares of NVIDIA (NVDA) currently cost **808.23 Swiss Francs (CHF)**.

This is based on the current stock price of $207.83 USD per share and the prevailing exchange rate.



### 5. Extra: Native Agentic Orchestration (The Modern SDK Way)

*While it is crucial to understand the manual ReAct loop, modern enterprise development rarely requires writing it from scratch. Frameworks and modern SDKs now handle tool orchestration natively.*

In [ ]:
# Create an agentic chat session WITH the tools attached natively
agentic_chat = client.chats.create(
    model="gemini-3-flash-preview",
    config=types.GenerateContentConfig(tools=agent_tools)
)

print(f"User: {prompt}\n")
print("--- Native SDK Agentic Execution ---\n")

# ===========================================================================
# 🚨 LECTURE NOTE: WHERE IS THE WHILE LOOP? 🚨
#
# Notice that there is no manual `while` loop here orchestrating the
# Thought -> Action -> Observation cycle like we had in the previous cell.
#
# Modern SDKs treat tool-use as a native capability. When we call `send_message()`
# below, the SDK automatically handles the entire ReAct loop under the hood:
#   1. It checks if the model requested a tool.
#   2. It halts text generation and automatically executes our Python function.
#   3. It packages the observation and feeds it back to the model.
#   4. It repeats this process automatically until the final goal is achieved.
# ===========================================================================

# Sending the prompt. Watch the console as the SDK automatically routes the tools!
response_native = agentic_chat.send_message(prompt)

print("--- Final Output to User ---")
print(response_native.text)

User: I want to buy 5 shares of NVIDIA right now. Exactly how much will that cost me in Swiss Francs (CHF) based on the current live market?

--- Native SDK Agentic Execution ---

⚙️ AGENT ACTION: Fetching live stock price for NVDA...
👀 AGENT OBSERVATION: NVDA is currently $207.83
⚙️ AGENT ACTION: Converting 1039.15 from USD to CHF...
👀 AGENT OBSERVATION: Conversion successful -> 808.13 CHF
--- Final Output to User ---
Based on the current live market price, here is the breakdown for buying 5 shares of NVIDIA (NVDA):

*   **Current Price per Share:** $207.83 USD
*   **Total Cost for 5 Shares:** $1,039.15 USD
*   **Total Cost in Swiss Francs:** **808.13 CHF**
